In [ ]:
!pip -q install datasets sentencepiece accelerate rouge-score

In [54]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
import os
import shutil
import json

import transformers
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModel, DataCollatorForSeq2Seq
from tqdm.auto import tqdm
from rouge_score import rouge_scorer
import torch.nn.functional as F
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

In [ ]:
print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)

In [ ]:
# GPU verification Kaggle code
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
train_df = pd.read_csv("/kaggle/input/datasets/irakozegaddiel/fineduqa-fiqa-data/fiqa_train.csv") 
val_df = pd.read_csv("/kaggle/input/datasets/irakozegaddiel/fineduqa-fiqa-data/fiqa_validation.csv")
test_df = pd.read_csv("/kaggle/input/datasets/irakozegaddiel/fineduqa-fiqa-data/fiqa_test.csv")

print(train_df.shape)
print(val_df.shape)
print(test_df.shape)

In [ ]:
# Load the tokenizer
MODEL_NAME = "google/flan-t5-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

For FLAN-T5, let's format the input as an instruction

In [ ]:
def format_question(question):
    return (
        "Answer the following financial literacy question: "
        + question
    )

Now, let's calculate token lengths on the training set

In [ ]:
train_input_lengths = train_df["question_clean"].apply(
    lambda x: len(
        tokenizer(format_question(x), add_special_tokens=True)["input_ids"]
    ))

train_target_lengths = train_df["answer_clean"].apply(
    lambda x: len(
        tokenizer(x, add_special_tokens=True)["input_ids"]
    ))

In [ ]:
# Calculate the full token-length statistics
def token_summary(lengths, name):
    print(f"\n{name}")
    print("Median:", int(np.median(lengths)))
    print("90th percentile:", int(np.percentile(lengths, 90)))
    print("95th percentile:", int(np.percentile(lengths, 95)))
    print("99th percentile:", int(np.percentile(lengths, 99)))
    print("Maximum:", int(np.max(lengths)))

token_summary(train_input_lengths, "INPUT TOKEN LENGTHS")
token_summary(train_target_lengths, "TARGET TOKEN LENGTHS")

In [ ]:
MAX_INPUT_LENGTH = 64
MAX_TARGET_LENGTH = 512

We choose 64, because our questions are very short: maximum 51 tokens, and 512 because for output because the answer's medianis  190, and the 90th percentile is 483.

Next, let's tokenize the three splits and convert them into HuggingFace datasets

In [ ]:
train_model_df = train_df[["question_clean", "answer_clean"]].copy()
val_model_df = val_df[["question_clean", "answer_clean"]].copy()
test_model_df = test_df[["question_clean", "answer_clean"]].copy()

In [ ]:
train_dataset = Dataset.from_pandas(train_model_df, preserve_index=False)
val_dataset = Dataset.from_pandas(val_model_df, preserve_index=False)
test_dataset = Dataset.from_pandas(test_model_df, preserve_index=False)

Next, we will define the preprocessing function

In [ ]:
def preprocess_function(examples):

    inputs = ["Answer the following financial literacy question: " + question
        for question in examples["question_clean"]
    ]

    model_inputs = tokenizer(inputs, max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=examples["answer_clean"], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [ ]:
tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=train_dataset.column_names)
tokenized_val = val_dataset.map(preprocess_function, batched=True, remove_columns=val_dataset.column_names)
tokenized_test = test_dataset.map(preprocess_function, batched=True, remove_columns=test_dataset.column_names)

In [ ]:
print(tokenized_train)
print(tokenized_val)
print(tokenized_test)

print("\nFirst tokenized example:")
print(tokenized_train[0])

## Model Training

### I. FLAN-T5 Baseline

In [ ]:
# Load the pre-trained model
MODEL_NAME = "google/flan-t5-base"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

print("Device:", device)

In [ ]:
# Define a generation function

def generate_answer(question):

    prompt = ("Answer the following financial literacy question: " + question)
    inputs = tokenizer(prompt, return_tensors="pt", max_length=MAX_INPUT_LENGTH,
        truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=MAX_TARGET_LENGTH, num_beams=4, early_stopping=True)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

Let's test it manually first: Before evaluating all 589 test rows, confirm that generation works, by generating answers to the first 5 questions.

In [ ]:
for i in range(5):

  sample_question = test_df.iloc[i]["question_clean"]
  reference_answer = test_df.iloc[i]["answer_clean"]

  baseline_answer = generate_answer(sample_question)

  print("QUESTION:")
  print(sample_question)

  print("\nREFERENCE ANSWER:")
  print(reference_answer)

  print("\nBASELINE ANSWER:")
  print(baseline_answer)

  print("============================================")

Now, let's Generate baseline predictions for the full test set, and attach the predictions.

In [ ]:
baseline_predictions = []

for question in tqdm(test_df["question_clean"], desc="Generating baseline answers"):
    prediction = generate_answer(question)
    baseline_predictions.append(prediction)

In [ ]:
baseline_results = test_df[["question_clean", "answer_clean"]].copy()
baseline_results["baseline_prediction"] = (baseline_predictions)

print(baseline_results.shape)

display(baseline_results[
        [
            "question_clean",
            "answer_clean",
            "baseline_prediction"
        ]].head())

Now, let me save the predictions on my google drive for an easy retrieval in the future.

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

In [ ]:
# We define the target folder
BASELINE_DIR = ("/content/drive/MyDrive/""ML Techniques I Summative/""Baseline Model")
os.makedirs(BASELINE_DIR, exist_ok=True)
print("Baseline folder:", BASELINE_DIR)

# Save there baseline predictions CSV there
baseline_file_path = os.path.join(BASELINE_DIR, "flan_t5_baseline_predictions.csv")
baseline_results.to_csv(baseline_file_path, index=False)

print("Saved successfully to:")
print(baseline_file_path)

**Now it's time for `EValuation`:**

First, we're gonna load the metrics, and Prepare the predictions and references

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

baseline_results = pd.read_csv("/content/drive/MyDrive/ML Techniques I Summative/"
    "Baseline Model/flan_t5_baseline_predictions.csv")

# Prepare predictions and references
predictions = (baseline_results["baseline_prediction"].astype(str).tolist())
references = (baseline_results["answer_clean"].astype(str).tolist())

print("Loaded rows:", len(baseline_results))
print("Predictions:", len(predictions))
print("References:", len(references))

After thatm let's compute ROUGE and BERTScore

In [ ]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for pred, ref in zip(predictions, references):
    scores = scorer.score(ref, pred)

    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

rouge1 = np.mean(rouge1_scores)
rouge2 = np.mean(rouge2_scores)
rougeL = np.mean(rougeL_scores)

print("ROUGE RESULTS")
print("ROUGE1:", rouge1)
print("ROUGE2:", rouge2)
print("ROUGEL:", rougeL)

In [ ]:
SEMANTIC_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

semantic_tokenizer = AutoTokenizer.from_pretrained(SEMANTIC_MODEL)
semantic_model = AutoModel.from_pretrained(SEMANTIC_MODEL).to("cuda")

semantic_model.eval()

In [ ]:
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state

    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()

    return (torch.sum(token_embeddings * mask, dim=1)
        / torch.clamp(mask.sum(dim=1), min=1e-9))

In [ ]:
# Pair-wise similarity calculation
def get_embeddings(texts, batch_size=16):
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i + batch_size]

        encoded = semantic_tokenizer(batch, padding=True, truncation=True,
            max_length=512, return_tensors="pt").to("cuda")

        with torch.no_grad():
            output = semantic_model(**encoded)

        embeddings = mean_pooling(output, encoded["attention_mask"])
        embeddings = F.normalize(embeddings, p=2, dim=1)

        all_embeddings.append(embeddings.cpu())

    return torch.cat(all_embeddings)

In [ ]:
prediction_embeddings = get_embeddings(predictions)
reference_embeddings = get_embeddings(references)

semantic_scores = F.cosine_similarity(prediction_embeddings, reference_embeddings)
semantic_similarity = semantic_scores.mean().item()

print("Mean Semantic Similarity:", semantic_similarity)

In [ ]:
baseline_metrics = {
    "model": "google/flan-t5-base",
    "rouge1": rouge1,
    "rouge2": rouge2,
    "rougeL": rougeL,
    "semantic_similarity": semantic_similarity
}

baseline_metrics_df = pd.DataFrame([baseline_metrics])
metrics_path = ("/content/drive/MyDrive/ML Techniques I Summative/Baseline Model/flan_T5_baseline_metrics.csv")
baseline_metrics_df.to_csv(metrics_path, index=False)

print("Saved to:")
print(metrics_path)

### II. FLAN-T5-Base finetuned

In [ ]:
MODEL_NAME = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

print("Fresh model loaded on:", device)

In [ ]:
print("Number of GPUs:", torch.cuda.device_count())

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

In [ ]:
# Sanity check: verify that the model produces a finite loss
sample_batch = data_collator([tokenized_train[i] for i in range(2)])

sample_batch = {
    key: value.to(device)
    for key, value in sample_batch.items()
}

model.eval()

with torch.no_grad():
    outputs = model(**sample_batch)

print("Sample loss:", outputs.loss.item())
print("Loss is finite:", torch.isfinite(outputs.loss).item())

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/flan_t5_financial_finetuned_fp32",

    num_train_epochs=3,
    learning_rate=5e-5,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    # We have 2 GPUs:
    # 2 examples × 2 GPUs × 2 accumulation = effective batch size 8
    gradient_accumulation_steps=2,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=100,

    # IMPORTANT: avoid the FP16 instability we saw in Colab
    fp16=False,
    bf16=False,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,
    report_to="none",

    seed=42
)

Next, let's create the trainer and start fine-tuning it

In [ ]:
# Create a new trainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    processing_class=tokenizer
)

print("Trainer created successfully.")

In [ ]:
# Start the training run
train_result = trainer.train()

After training, we evaluate the model that the **trainer** loaded as the best checkpoint

In [ ]:
eval_results = trainer.evaluate()

print("Final evaluation:")
print(eval_results)

In [ ]:
print("Best checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation loss:", trainer.state.best_metric)

Now let's save the best model separetly

In [ ]:
BEST_MODEL_DIR = "/kaggle/working/flan_t5_financial_finetuned_best"

trainer.save_model(BEST_MODEL_DIR)
tokenizer.save_pretrained(BEST_MODEL_DIR)

print("Best model saved to:", BEST_MODEL_DIR)

Let's now move to **test-set** inference

In [ ]:
model = trainer.model
model.eval()

print("Fine-tuned best model ready for inference.")

In [ ]:
def generate_finetuned_answer(question):

    prompt = "Answer the following financial literacy question: " + question
    inputs = tokenizer(prompt, return_tensors="pt", max_length=MAX_INPUT_LENGTH, truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=MAX_TARGET_LENGTH, num_beams=4, early_stopping=True)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

In [ ]:
for i in range(3):

    question = test_df.iloc[i]["question_clean"]
    reference = test_df.iloc[i]["answer_clean"]
    prediction = generate_finetuned_answer(question)

    print("QUESTION:")
    print(question)

    print("\nREFERENCE:")
    print(reference)

    print("\nFINE-TUNED ANSWER:")
    print(prediction)

    print("\n" + "=" * 80 + "\n")

Those three outputs are clearly degenerate: the model is looping phrases and giving weak generic answers.

To figure out why, let's first measure the validation loss of the baseline model. That comparison will tell us whether fine-tuning actually made the underlying objective worse.

What we are looking for is straightforward:
- If the fresh baseline validation loss is substantially below 6.40, then the first fine-tuning run clearly damaged the pretrained model. We will run Experiment 2 with a much gentler learning rate and warm-up.
- If the fresh baseline loss is above 6.40, then fine-tuning improved token-level loss despite producing poor generation, and we need to investigate the generation/training-target behavior rather than immediately blaming the learning rate.

In [ ]:
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

base_val_loader = DataLoader(tokenized_val, batch_size=2, shuffle=False, collate_fn=data_collator)

base_model.eval()

total_loss = 0.0
num_batches = 0

with torch.no_grad():
    for batch in tqdm(base_val_loader, desc="Evaluating fresh baseline"):
        batch = {
            key: value.to(BASE_DEVICE)
            for key, value in batch.items()
        }

        outputs = base_model(**batch)

        total_loss += outputs.loss.item()
        num_batches += 1

baseline_val_loss = total_loss / num_batches

print("Fresh baseline validation loss:", baseline_val_loss)
print("Fine-tuned validation loss:", 6.400600433349609)

For a fair comparison, let's manually evaluate the fine-tuned model using exactly the same procedure we used for the baseline.

In [ ]:
FT_DEVICE = torch.device("cuda:0")

finetuned_model = trainer.model
finetuned_model.to(FT_DEVICE)
finetuned_model.eval()

finetuned_val_loader = DataLoader(tokenized_val, batch_size=2, shuffle=False, collate_fn=data_collator)

total_loss = 0.0
num_batches = 0

with torch.no_grad():
    for batch in tqdm(
        finetuned_val_loader,
        desc="Evaluating fine-tuned model manually"
    ):
        batch = {
            key: value.to(FT_DEVICE)
            for key, value in batch.items()
        }

        outputs = finetuned_model(**batch)

        total_loss += outputs.loss.item()
        num_batches += 1

finetuned_manual_val_loss = total_loss / num_batches

print("Fresh baseline validation loss:", baseline_val_loss)
print("Fine-tuned manual validation loss:", finetuned_manual_val_loss)

We notice that the fine-tuned model actually improved the validation loss. So, the problem could be the decoding/generation behavior, especially the repetition.

In [ ]:
def generate_answer_with_controls(model, question, device):

    prompt = (
        "Answer the following financial literacy question: "
        + question
    )

    inputs = tokenizer(prompt, return_tensors="pt", max_length=MAX_INPUT_LENGTH, truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,

            # Keep deterministic beam search
            num_beams=4,
            early_stopping=True,

            # Prevent the looping we observed
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,

            # Enough room for a useful financial answer
            max_new_tokens=256
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

In [ ]:
for i in range(3):

    question = val_df.iloc[i]["question_clean"]
    reference = val_df.iloc[i]["answer_clean"]

    baseline_answer = generate_answer_with_controls(base_model, question, BASE_DEVICE)

    finetuned_answer = generate_answer_with_controls(finetuned_model, question, FT_DEVICE)

    print("QUESTION:")
    print(question)

    print("\nREFERENCE:")
    print(reference)

    print("\nBASELINE:")
    print(baseline_answer)

    print("\nFINE-TUNED:")
    print(finetuned_answer)

    print("\n" + "=" * 80 + "\n")

The repetition controls fixed the looping, but the fine-tuned answers are still not good enough. I would not generate the full 589 test predictions yet.

The problems are clear: the first answer is vague and unfinished, the second contains factual confusion (“Vanguard ETF is a mutual fund…”), and the third is generic rather than really answering the situation. So the model's validation loss improved, but generation quality did not improve enough.

The next clean experiment should be a lower learning rate, while keeping everything else exactly the same. That gives us a controlled experiment rather than changing several things at once.

#### Experiment2

In [ ]:
MODEL_NAME = "google/flan-t5-base"

model_exp2 = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)
model_exp2.to(device)

data_collator_exp2 = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_exp2)

print("Fresh model for Experiment 2 loaded on:", device)

In [ ]:
# Create the training arguments
training_args_exp2 = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/flan_t5_exp2_lr1e-5",

    num_train_epochs=3,

    # Only experimental change
    learning_rate=1e-5,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=100,

    fp16=False,
    bf16=False,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,
    report_to="none",
    seed=42
)

In [ ]:
trainer_exp2 = Seq2SeqTrainer(
    model=model_exp2,
    args=training_args_exp2,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator_exp2,
    processing_class=tokenizer
)

print("Experiment 2 trainer ready.")

In [ ]:
train_result_exp2 = trainer_exp2.train()

Training finished successfully. Experiment 2 is numerically stable, but based on the Trainer-reported validation loss it performed a little worse than Experiment 1. However, that does not yet mean we reject Experiment 2, because we care especially about whether the lower learning rate preserved better answer quality.

Next step is to calculate Experiment 2 validation loss manually.

In [ ]:
EXP2_DEVICE = torch.device("cuda:0")

exp2_model = trainer_exp2.model
exp2_model.to(EXP2_DEVICE)
exp2_model.eval()

exp2_val_loader = DataLoader(tokenized_val, batch_size=2, shuffle=False, collate_fn=data_collator_exp2)

total_loss = 0.0
num_batches = 0

with torch.no_grad():
    for batch in tqdm(
        exp2_val_loader,
        desc="Evaluating Experiment 2 manually"
    ):
        batch = {
            key: value.to(EXP2_DEVICE)
            for key, value in batch.items()
        }

        outputs = exp2_model(**batch)

        total_loss += outputs.loss.item()
        num_batches += 1

exp2_manual_val_loss = total_loss / num_batches

print("Baseline validation loss:     ", baseline_val_loss)
print("Experiment 1 validation loss:", finetuned_manual_val_loss)
print("Experiment 2 validation loss:", exp2_manual_val_loss)

In [ ]:
# Save Experiment 2
EXP2_MODEL_DIR = "/kaggle/working/flan_t5_exp2_lr1e-5_best"

trainer_exp2.save_model(EXP2_MODEL_DIR)
tokenizer.save_pretrained(EXP2_MODEL_DIR)

print("Experiment 2 saved to:", EXP2_MODEL_DIR)

Let's now use the same 3 validation questions as before and compare all three models with the exact same decoding settings.

In [ ]:
for i in range(3):

    question = val_df.iloc[i]["question_clean"]
    reference = val_df.iloc[i]["answer_clean"]

    baseline_answer = generate_answer_with_controls(base_model, question, BASE_DEVICE)
    exp1_answer = generate_answer_with_controls(finetuned_model, question, FT_DEVICE)
    exp2_answer = generate_answer_with_controls(exp2_model, question, EXP2_DEVICE)

    print("QUESTION:")
    print(question)

    print("\nREFERENCE:")
    print(reference)

    print("\nBASELINE:")
    print(baseline_answer)

    print("\nEXPERIMENT 1 - LR 5e-5:")
    print(exp1_answer)

    print("\nEXPERIMENT 2 - LR 1e-5:")
    print(exp2_answer)

    print("\n" + "=" * 100 + "\n")

Experiment 2 is better qualitatively than Experiment 1, but it is still not good enough for the final test evaluation.

The next issue to fix is the **one-question-to-many-answers training structure**. Our 4,852 training rows represent only 1,592 unique questions, meaning the model sees the exact same input paired with different outputs. Around 70% of the training questions have multiple answers. For generative fine-tuning, that can encourage the model toward vague compromise answers like the ones we're seeing above.

#### Experiment 3: One representative answer per question

In [ ]:
# Create one representative training answer per unique question

train_single = (
    train_df
    .sort_values(["question_clean", "answer_words"])
    .groupby("question_clean", group_keys=False)
    .apply(lambda group: group.iloc[len(group) // 2])
    .reset_index(drop=True)
)

print("Original training rows:", len(train_df))
print("Single-reference training rows:", len(train_single))
print("Unique questions:", train_single["question_clean"].nunique())

print("\nAnswer length statistics:")
print(train_single["answer_words"].describe())

Let's first create and tokenize the training dataset for this experiment specifically.

In [ ]:
train_single_model_df = train_single[["question_clean", "answer_clean"]].copy()
train_single_dataset = Dataset.from_pandas(train_single_model_df, preserve_index=False)

tokenized_train_single = train_single_dataset.map(preprocess_function, batched=True,
    remove_columns=train_single_dataset.column_names)

print(tokenized_train_single)

In [ ]:
# Load a completely fresh FLAN-T5-base
model_exp3 = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")
model_exp3.to(device)

data_collator_exp3 = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model_exp3)
print("Fresh Experiment 3 model loaded on:", device)

In [ ]:
# Configure the experiment 
training_args_exp3 = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/flan_t5_exp3_single_reference",

    num_train_epochs=3,
    learning_rate=1e-5,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=2,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_strategy="steps",
    logging_steps=50,

    fp16=False,
    bf16=False,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    save_total_limit=2,
    report_to="none",

    seed=42
)

In [ ]:
trainer_exp3 = Seq2SeqTrainer(
    model=model_exp3,
    args=training_args_exp3,
    train_dataset=tokenized_train_single,
    eval_dataset=tokenized_val,
    data_collator=data_collator_exp3,
    processing_class=tokenizer
)

print("Experiment 3 trainer ready.")

In [ ]:
train_result_exp3 = trainer_exp3.train()

> **Note on Experiment 3 checkpoint saving:**  
> Training completed successfully for all 3 epochs. However, after the final epoch, an error occurred while the Hugging Face `Trainer` attempted to save the optimizer state as part of the checkpoint. The trained model weights were still available in memory and were successfully saved separately using `save_pretrained()` in `safetensors` format. Therefore, the error affected checkpoint/optimizer-state saving only and did not invalidate the completed training run or the final model weights.

In [ ]:
import os
import shutil

# Remove the incomplete checkpoint created when optimizer saving failed
failed_checkpoint = ("/kaggle/working/flan_t5_exp3_single_reference/checkpoint-597")

shutil.rmtree(failed_checkpoint, ignore_errors=True)

# Save only the trained model + tokenizer
EXP3_MODEL_DIR = "/kaggle/working/flan_t5_exp3_final_model"

model_exp3.save_pretrained(EXP3_MODEL_DIR, safe_serialization=True)
tokenizer.save_pretrained(EXP3_MODEL_DIR)

print("Experiment 3 saved successfully.")
print(os.listdir(EXP3_MODEL_DIR))

Next, calculate Experiment 3's manual validation loss using the exact same single-GPU procedure as before.

In [ ]:
EXP3_DEVICE = torch.device("cuda:0")

model_exp3.to(EXP3_DEVICE)
model_exp3.eval()

exp3_val_loader = DataLoader(tokenized_val, batch_size=2, shuffle=False, collate_fn=data_collator_exp3)

total_loss = 0.0
num_batches = 0

with torch.no_grad():
    for batch in tqdm(
        exp3_val_loader,
        desc="Evaluating Experiment 3 manually"
    ):
        batch = {
            key: value.to(EXP3_DEVICE)
            for key, value in batch.items()
        }

        outputs = model_exp3(**batch)

        total_loss += outputs.loss.item()
        num_batches += 1

exp3_manual_val_loss = total_loss / num_batches

print("Baseline validation loss:     ", baseline_val_loss)
print("Experiment 1 validation loss:", finetuned_manual_val_loss)
print("Experiment 2 validation loss:", exp2_manual_val_loss)
print("Experiment 3 validation loss:", exp3_manual_val_loss)

In [ ]:
for i in range(3):

    question = val_df.iloc[i]["question_clean"]
    reference = val_df.iloc[i]["answer_clean"]

    exp3_answer = generate_answer_with_controls(model_exp3, question, EXP3_DEVICE)

    print("QUESTION:")
    print(question)

    print("\nREFERENCE:")
    print(reference)

    print("\nEXPERIMENT 3 - SINGLE REFERENCE, LR 1e-5:")
    print(exp3_answer)

    print("\n" + "=" * 100 + "\n")

Experiment 1 achieved the lowest validation loss, but qualitative evaluation revealed repetitive and poorly formed responses. Reducing the learning rate in Experiment 2 slightly increased validation loss but improved generation stability and coherence. Experiment 3 tested a single-reference training formulation but did not improve answer quality. Therefore, Experiment 2 was selected as the final model based on the combined quantitative and qualitative evidence.

In [ ]:
import os
import shutil

# Make sure Experiment 2 is still in memory BEFORE deleting checkpoints
if "exp2_model" not in globals():
    raise RuntimeError(
        "STOP: exp2_model is not in memory. Do not delete checkpoints."
    )

print("✅ Experiment 2 model is still in memory.")

# Old Trainer checkpoint folders are no longer needed
folders_to_delete = [
    "/kaggle/working/flan_t5_financial_finetuned_fp32",   # Exp 1 checkpoints
    "/kaggle/working/flan_t5_exp2_lr1e-5",                # Exp 2 checkpoints
    "/kaggle/working/flan_t5_exp3_single_reference",      # Exp 3 checkpoints
    "/kaggle/working/flan_t5_exp2_final_model"            # corrupted save
]

for folder in folders_to_delete:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print("Deleted:", folder)

# Show remaining free disk space
total, used, free = shutil.disk_usage("/kaggle/working")

print(f"\nFree space: {free / (1024**3):.2f} GB")

In [ ]:
EXP2_FINAL_DIR = "/kaggle/working/flan_t5_exp2_final"

exp2_model.save_pretrained(EXP2_FINAL_DIR, safe_serialization=True)

tokenizer.save_pretrained(EXP2_FINAL_DIR)

print("✅ Experiment 2 saved successfully.")
print(os.listdir(EXP2_FINAL_DIR))

In [ ]:
FINAL_DEVICE = torch.device("cuda:0")

# Move Experiment 2 off the GPU temporarily
exp2_model.to("cpu")
torch.cuda.empty_cache()

# Load a fresh untouched FLAN-T5 baseline
baseline_final_model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base").to(FINAL_DEVICE)
baseline_final_model.eval()

baseline_final_predictions = []

for question in tqdm(test_df["question_clean"], desc="Generating final baseline predictions"):
    
    answer = generate_answer_with_controls(baseline_final_model, question, FINAL_DEVICE)
    baseline_final_predictions.append(answer)

# Save predictions
baseline_final_df = test_df[["question_clean", "answer_clean"]].copy()
baseline_final_df["baseline_prediction"] = baseline_final_predictions

BASELINE_FINAL_PATH = ("/kaggle/working/final_baseline_test_predictions.csv")
baseline_final_df.to_csv(BASELINE_FINAL_PATH, index=False)

print("Predictions generated:", len(baseline_final_df))
print("Saved to:", BASELINE_FINAL_PATH)

baseline_final_df[["question_clean", "baseline_prediction"]].head(3)

In [ ]:
# Move baseline off GPU
baseline_final_model.to("cpu")
torch.cuda.empty_cache()

# Move Experiment 2 to GPU
exp2_model.to(FINAL_DEVICE)
exp2_model.eval()

exp2_final_predictions = []

for question in tqdm(test_df["question_clean"], desc="Generating Experiment 2 test predictions"):
    
    answer = generate_answer_with_controls(exp2_model, question, FINAL_DEVICE)
    exp2_final_predictions.append(answer)

# Save predictions
exp2_final_df = test_df[["question_clean", "answer_clean"]].copy()
exp2_final_df["exp2_prediction"] = exp2_final_predictions

EXP2_FINAL_PRED_PATH = ("/kaggle/working/final_exp2_test_predictions.csv")
exp2_final_df.to_csv(EXP2_FINAL_PRED_PATH, index=False)

print("Predictions generated:", len(exp2_final_df))
print("Saved to:", EXP2_FINAL_PRED_PATH)

exp2_final_df[["question_clean", "exp2_prediction"]].head(3)

In [ ]:
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def calculate_rouge(references, predictions):
    rouge1_scores = []
    rouge2_scores = []
    rougeL_scores = []

    for reference, prediction in tqdm(zip(references, predictions), total=len(references)):
        
        scores = scorer.score(str(reference), str(prediction))

        rouge1_scores.append(scores["rouge1"].fmeasure)
        rouge2_scores.append(scores["rouge2"].fmeasure)
        rougeL_scores.append(scores["rougeL"].fmeasure)

    return {
        "ROUGE-1": np.mean(rouge1_scores),
        "ROUGE-2": np.mean(rouge2_scores),
        "ROUGE-L": np.mean(rougeL_scores)
    }


references = test_df["answer_clean"].tolist()

baseline_rouge = calculate_rouge(references, baseline_final_df["baseline_prediction"].tolist())
exp2_rouge = calculate_rouge(references, exp2_final_df["exp2_prediction"].tolist())

rouge_comparison = pd.DataFrame({
    "Metric": ["ROUGE-1", "ROUGE-2", "ROUGE-L"],
    "Baseline": [
        baseline_rouge["ROUGE-1"],
        baseline_rouge["ROUGE-2"],
        baseline_rouge["ROUGE-L"]
    ],
    "Experiment 2": [
        exp2_rouge["ROUGE-1"],
        exp2_rouge["ROUGE-2"],
        exp2_rouge["ROUGE-L"]
    ]
})

rouge_comparison["Difference"] = (rouge_comparison["Experiment 2"]- rouge_comparison["Baseline"])
display(rouge_comparison)

In [ ]:
SEMANTIC_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SEMANTIC_DEVICE = torch.device("cuda:0")

# Free GPU memory from FLAN-T5
exp2_model.to("cpu")
torch.cuda.empty_cache()

semantic_tokenizer = AutoTokenizer.from_pretrained(SEMANTIC_MODEL)
semantic_model = AutoModel.from_pretrained(SEMANTIC_MODEL).to(SEMANTIC_DEVICE)

semantic_model.eval()

def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    return torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    ) / torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )


def encode_texts(texts, batch_size=16):
    embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc="Encoding"):
        batch_texts = texts[i:i + batch_size]

        encoded = semantic_tokenizer(batch_texts, padding=True, truncation=True,
            max_length=512, return_tensors="pt")

        encoded = {
            key: value.to(SEMANTIC_DEVICE)
            for key, value in encoded.items()
        }

        with torch.no_grad():
            output = semantic_model(**encoded)

        batch_embeddings = mean_pooling(output, encoded["attention_mask"])
        batch_embeddings = F.normalize(batch_embeddings, p=2, dim=1)

        embeddings.append(batch_embeddings.cpu())

    return torch.cat(embeddings, dim=0)


references = test_df["answer_clean"].astype(str).tolist()

baseline_predictions = (
    baseline_final_df["baseline_prediction"]
    .astype(str)
    .tolist()
)

exp2_predictions = (
    exp2_final_df["exp2_prediction"]
    .astype(str)
    .tolist()
)


print("Encoding references...")
reference_embeddings = encode_texts(references)

print("\nEncoding baseline predictions...")
baseline_embeddings = encode_texts(baseline_predictions)

print("\nEncoding Experiment 2 predictions...")
exp2_embeddings = encode_texts(exp2_predictions)

baseline_similarities = F.cosine_similarity(reference_embeddings, baseline_embeddings, dim=1)
exp2_similarities = F.cosine_similarity(reference_embeddings, exp2_embeddings, dim=1)

baseline_semantic = baseline_similarities.mean().item()
exp2_semantic = exp2_similarities.mean().item()

print("\nFinal Semantic Similarity")
print("Baseline:     ", baseline_semantic)
print("Experiment 2:", exp2_semantic)
print("Difference:   ", exp2_semantic - baseline_semantic)

Fine-tuning substantially improved alignment with domain-specific reference answers, but higher automatic scores did not eliminate factual inaccuracies. This highlights the importance of qualitative error analysis alongside ROUGE and semantic similarity.

In [ ]:
final_metrics = pd.DataFrame({
    "Metric": [
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "Semantic Similarity"
    ],
    "Baseline": [
        baseline_rouge["ROUGE-1"],
        baseline_rouge["ROUGE-2"],
        baseline_rouge["ROUGE-L"],
        baseline_semantic
    ],
    "Fine-tuned Experiment 2": [
        exp2_rouge["ROUGE-1"],
        exp2_rouge["ROUGE-2"],
        exp2_rouge["ROUGE-L"],
        exp2_semantic
    ]
})

final_metrics["Absolute Improvement"] = (final_metrics["Fine-tuned Experiment 2"] - final_metrics["Baseline"])
FINAL_METRICS_PATH = "/kaggle/working/final_model_metrics.csv"
final_metrics.to_csv(FINAL_METRICS_PATH, index=False)

display(final_metrics)
print("Saved to:", FINAL_METRICS_PATH)

#### Error Analysis

In [ ]:
# Build per-example error-analysis table
error_analysis_df = test_df[["question_clean", "answer_clean"]].copy()

error_analysis_df["baseline_prediction"] = (baseline_final_df["baseline_prediction"].values)
error_analysis_df["exp2_prediction"] = (exp2_final_df["exp2_prediction"].values)

# Per-example semantic similarity
error_analysis_df["baseline_semantic"] = (baseline_similarities.cpu().numpy())
error_analysis_df["exp2_semantic"] = (exp2_similarities.cpu().numpy())

# Per-example ROUGE-L for Experiment 2
exp2_rougeL = []

for reference, prediction in zip(error_analysis_df["answer_clean"], error_analysis_df["exp2_prediction"]):
    
    score = scorer.score(str(reference), str(prediction))
    exp2_rougeL.append(score["rougeL"].fmeasure)

error_analysis_df["exp2_rougeL"] = exp2_rougeL
print(error_analysis_df.shape)

display(
    error_analysis_df[
        [
            "question_clean",
            "answer_clean",
            "baseline_prediction",
            "exp2_prediction",
            "exp2_semantic",
            "exp2_rougeL"
        ]
    ].head()
)

In [ ]:
question_scores = (
    error_analysis_df
    .groupby("question_clean", as_index=False)
    .agg(
        exp2_prediction=("exp2_prediction", "first"),
        mean_semantic=("exp2_semantic", "mean"),
        mean_rougeL=("exp2_rougeL", "mean"),
        reference_count=("answer_clean", "count")
    )
)

print("Unique questions:", len(question_scores))
display(question_scores.head())

In [ ]:
# Sort unique questions by semantic similarity
question_scores_sorted = (question_scores.sort_values("mean_semantic").reset_index(drop=True))

# 5 lowest-performing questions
low_examples = question_scores_sorted.head(5).copy()
low_examples["performance_group"] = "Low"

# 5 questions around the median
middle = len(question_scores_sorted) // 2

medium_examples = question_scores_sorted.iloc[middle - 2 : middle + 3].copy()
medium_examples["performance_group"] = "Medium"

# 5 highest-performing questions
high_examples = question_scores_sorted.tail(5).copy()
high_examples["performance_group"] = "High"

# Combine the 15 selected questions
error_sample = pd.concat([low_examples, medium_examples, high_examples], ignore_index=True)

display(
    error_sample[
        [
            "performance_group",
            "question_clean",
            "exp2_prediction",
            "mean_semantic",
            "mean_rougeL",
            "reference_count"
        ]
    ]
)

In [ ]:
low_questions = error_sample.loc[error_sample["performance_group"] == "Low", "question_clean"].tolist()

for i, question in enumerate(low_questions, start=1):
    rows = error_analysis_df[error_analysis_df["question_clean"] == question]
    prediction = rows["exp2_prediction"].iloc[0]

    print("=" * 100)
    print(f"LOW EXAMPLE {i}")

    print("\nQUESTION:")
    print(question)

    print("\nEXPERIMENT 2 PREDICTION:")
    print(prediction)

    print("\nREFERENCE ANSWER(S):")
    
    for j, reference in enumerate(rows["answer_clean"].tolist(), start=1):
        print(f"\nReference {j}:")
        print(reference)

    print("\nMean semantic similarity:", rows["exp2_semantic"].mean())
    print("Mean ROUGE-L:", rows["exp2_rougeL"].mean())
    print()

| # | Main error category | Secondary issue | Short analysis |
|---|---|---|---|
| 1 | **Irrelevant / non-answer** | Generic/vague | The ETF-fee question is not answered at all. The model responds with a conversational filler instead of explaining how fees affect NAV/market price. |
| 2 | **Irrelevant / non-answer** | Generic/vague | The prediction gives no insurance or legal explanation and does not address the scenario. |
| 3 | **Incorrect answer** | Misunderstanding of intent | The user is asking how to position against a possible Canadian real-estate crash, but the model recommends investing in Canadian real estate—the opposite direction of the question. |
| 4 | **Irrelevant / incomplete** | Generic financial language | The answer discusses whether to sell a mutual fund but never explains why bond funds were not experiencing major selloffs. |
| 5 | **Irrelevant / non-answer** | Generic/vague | The model does not explain how RSUs are treated during loan qualification and instead produces another filler response. |

In [ ]:
medium_questions = error_sample.loc[error_sample["performance_group"] == "Medium", "question_clean"].tolist()

for i, question in enumerate(medium_questions, start=1):
    rows = error_analysis_df[error_analysis_df["question_clean"] == question]
    prediction = rows["exp2_prediction"].iloc[0]

    print("=" * 100)
    print(f"MEDIUM EXAMPLE {i}")

    print("\nQUESTION:")
    print(question)

    print("\nEXPERIMENT 2 PREDICTION:")
    print(prediction)

    print("\nREFERENCE ANSWER(S):")
    
    for j, reference in enumerate(rows["answer_clean"].tolist(), start=1):
        print(f"\nReference {j}:")
        print(reference)

    print("\nMean semantic similarity:", rows["exp2_semantic"].mean())
    print("Mean ROUGE-L:", rows["exp2_rougeL"].mean())
    print()

| # | Main error category | Secondary issue | Short analysis |
|---|---|---|---|
| 1 | **Incomplete / generic** | Misses context | The model recognizes that the user could invest in stocks, bonds, or funds, but gives very generic advice and ignores important context from the references such as goals, emergency reserves, geographic risk, and asset allocation. |
| 2 | **Generic / weak answer** | Possible hallucination | It understands that the question asks about teaching children to save, but it does not recommend any of the concrete games found in the references and introduces a vague game called `"Saving Money"` without support. |
| 3 | **Partially relevant but inaccurate** | Factual inconsistency | It correctly recognizes that interest-free loans may exist, but then confusingly suggests getting a mortgage with a lower interest rate, which is not the same as a no-interest loan. |
| 4 | **Irrelevant / non-answer** | Generic fallback | It completely fails to name or describe software that can work without linking online banking, despite several valid examples being present in the references. |
| 5 | **Incorrect explanation** | Misunderstands economics | It recognizes that currency appreciation can create problems, but the explanation is fundamentally wrong. An appreciating currency does not become “worthless”; the key issue is the effect on exports, imports, competitiveness, and employment. |

Medium-scoring responses generally remained on the correct financial topic but often lacked specificity, omitted important context, or contained factual inconsistencies. This shows that moderate semantic similarity may reflect topical overlap without guaranteeing answer correctness.

In [ ]:
high_questions = error_sample.loc[error_sample["performance_group"] == "High", "question_clean"].tolist()

for i, question in enumerate(high_questions, start=1):
    rows = error_analysis_df[error_analysis_df["question_clean"] == question]
    prediction = rows["exp2_prediction"].iloc[0]

    print("=" * 100)
    print(f"HIGH EXAMPLE {i}")

    print("\nQUESTION:")
    print(question)

    print("\nEXPERIMENT 2 PREDICTION:")
    print(prediction)

    print("\nREFERENCE ANSWER(S):")
    
    for j, reference in enumerate(rows["answer_clean"].tolist(), start=1):
        print(f"\nReference {j}:")
        print(reference)

    print("\nMean semantic similarity:", rows["exp2_semantic"].mean())
    print("Mean ROUGE-L:", rows["exp2_rougeL"].mean())
    print()

| # | Main error category | Secondary issue | Short analysis |
|---|---|---|---|
| 1 | **Partially correct but factually inaccurate** | Concept confusion | The model correctly suggests a mutual fund as a way to invest, but incorrectly says the S&P 500 itself is a stock. It is a stock-market index. |
| 2 | **Irrelevant / incomplete** | Generic fallback | The model never actually answers whether a business can refuse credit cards. Instead, it drifts into bank loans and interest rates. |
| 3 | **Incomplete / vague** | Misses key risk factors | The response recognizes some uncertainty around taking a mortgage in Greece, but fails to explain the major issues in the references: political risk, currency risk, property-value risk, and loan denomination. |
| 4 | **Incorrect answer** | Contradicts the question | The model recommends using a personal account when the question asks why a business checking account may be preferable. This directly conflicts with the reference rationale. |
| 5 | **Incorrect / irrelevant explanation** | Fails to explain risk | The model never explains why mortgages tend to be cheaper than gold-backed loans. The key reference explanation is lender risk and the characteristics of the collateral. |

High-scoring responses generally exhibited strong topical similarity to the references, but manual inspection revealed persistent factual inaccuracies, vague explanations, and occasional contradictions. This indicates that semantic similarity and ROUGE can capture topical overlap without fully measuring factual correctness or answer usefulness.

### Outputfiles permanent saving

In [ ]:
important_paths = [
    "/kaggle/working/flan_t5_exp2_final",
    "/kaggle/working/flan_t5_exp3_final_model",
    "/kaggle/working/flan_t5_financial_finetuned_best",
    "/kaggle/working/final_baseline_test_predictions.csv",
    "/kaggle/working/final_exp2_test_predictions.csv",
    "/kaggle/working/final_model_metrics.csv"
]

for path in important_paths:
    status = "✅ EXISTS" if os.path.exists(path) else "❌ MISSING"
    print(status, path)

In [ ]:
model_folders = [
    "/kaggle/working/flan_t5_exp2_final",
    "/kaggle/working/flan_t5_exp3_final_model",
    "/kaggle/working/flan_t5_financial_finetuned_best"
]

for folder in model_folders:
    if os.path.isdir(folder):
        zip_path = shutil.make_archive(folder, "zip", folder)
        print("Created:", zip_path)

In [ ]:
import os
from IPython.display import FileLink, display

os.chdir("/kaggle/working")

files_to_download = [
    "flan_t5_exp2_final.zip",
    "flan_t5_exp3_final_model.zip",
    "flan_t5_financial_finetuned_best.zip",
    "final_baseline_test_predictions.csv",
    "final_exp2_test_predictions.csv",
    "final_model_metrics.csv"
]

for file in files_to_download:
    display(FileLink(file))

### Reload the models for a new session

We'll first need to verify that all the previously saved outputs were mounted sucessfully in this new session as inputs.

In [8]:
targets = ["flan_t5_exp2_final.zip", "final_model_metrics.csv", "final_exp2_test_predictions.csv", "final_baseline_test_predictions.csv"]

for root, dirs, files in os.walk("/kaggle/input"):
    for file in targets:
        if file in files:
            print(file, "->", os.path.join(root, file))

flan_t5_exp2_final.zip -> /kaggle/input/notebooks/irakozegaddiel/02-flan-t5-modeling/flan_t5_exp2_final.zip
final_model_metrics.csv -> /kaggle/input/notebooks/irakozegaddiel/02-flan-t5-modeling/final_model_metrics.csv
final_exp2_test_predictions.csv -> /kaggle/input/notebooks/irakozegaddiel/02-flan-t5-modeling/final_exp2_test_predictions.csv
final_baseline_test_predictions.csv -> /kaggle/input/notebooks/irakozegaddiel/02-flan-t5-modeling/final_baseline_test_predictions.csv


In [6]:
for root, dirs, files in os.walk("/kaggle/input"):
    if "model.safetensors" in files:
        print("Model found ->", root)

Model found -> /kaggle/input/notebooks/irakozegaddiel/02-flan-t5-modeling/flan_t5_exp3_final_model
Model found -> /kaggle/input/notebooks/irakozegaddiel/02-flan-t5-modeling/flan_t5_exp2_lr1e-5_best
Model found -> /kaggle/input/notebooks/irakozegaddiel/02-flan-t5-modeling/flan_t5_financial_finetuned_best
Model found -> /kaggle/input/notebooks/irakozegaddiel/02-flan-t5-modeling/flan_t5_exp2_final


In [9]:
MODEL_PATH = "/kaggle/input/notebooks/irakozegaddiel/02-flan-t5-modeling/flan_t5_exp2_final"
DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
exp2_model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH).to(DEVICE)
exp2_model.eval()

print("Device:", DEVICE)
print("Experiment 2 loaded successfully.")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Device: cuda:0
Experiment 2 loaded successfully.


In [13]:
# Restore our final generation function

MAX_INPUT_LENGTH = 64

def generate_answer_with_controls(model, question, device):
    prompt = "Answer the following financial literacy question: " + question
    inputs = tokenizer(prompt, return_tensors="pt", max_length=MAX_INPUT_LENGTH, truncation=True).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            num_beams=4,
            early_stopping=True,
            repetition_penalty=1.2,
            no_repeat_ngram_size=3,
            max_new_tokens=256
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

Now, the next essential step is to do **out-of-domain (OOD) evaluation.** We will use 10 manually written OOD questions: 5 completely unrelated to finance and 5 finance-related but outside our intended scope.

In [ ]:
ood_questions = [
    {"category": "Non-finance", "question": "How does photosynthesis work?"},
    {"category": "Non-finance", "question": "How do I fix an IndexError in Python?"},
    {"category": "Non-finance", "question": "What causes malaria?"},
    {"category": "Non-finance", "question": "How do I cook rice properly?"},
    {"category": "Non-finance", "question": "Why is the sky blue?"},
    {"category": "Out-of-scope finance", "question": "Which stock should I buy today for the highest short-term return?"},
    {"category": "Out-of-scope finance", "question": "What options strategy should I use to maximize profit this week?"},
    {"category": "Out-of-scope finance", "question": "How should I calculate deferred tax liabilities for a multinational corporation?"},
    {"category": "Out-of-scope finance", "question": "Should I short Bitcoin before the next Federal Reserve announcement?"},
    {"category": "Out-of-scope finance", "question": "How do I value a company using a leveraged buyout model?"}
]

ood_df = pd.DataFrame(ood_questions)
ood_predictions = []

for question in tqdm(ood_df["question"], desc="Evaluating OOD questions"):
    answer = generate_answer_with_controls(exp2_model, question, DEVICE)
    ood_predictions.append(answer)

ood_df["prediction"] = ood_predictions

In [18]:
for i, row in ood_df.iterrows():
    print("=" * 100)
    print(f"OOD EXAMPLE {i + 1}")

    print("\nCATEGORY:", row["category"])

    print("\nQUESTION:", row["question"])

    print("\nPREDICTION:")
    print(row["prediction"])

    print()

OOD EXAMPLE 1

CATEGORY: Non-finance

QUESTION: How does photosynthesis work?

PREDICTION:
The photosynthesis process is a complex process that requires a lot of energy. It takes a long time for the photosynthesis to take place, but it's a very simple process. If you want to make a profit, you need to be able to produce a large amount of energy in order to get a good return on your investment.

OOD EXAMPLE 2

CATEGORY: Non-finance

QUESTION: How do I fix an IndexError in Python?

PREDICTION:
I'm not sure if it's a bug or a security issue, but if you're looking for a way to fix an index error in Python, this is the answer.

OOD EXAMPLE 3

CATEGORY: Non-finance

QUESTION: What causes malaria?

PREDICTION:
I'm not sure if it's a virus or a parasite, but I've been told that malaria is a very common disease in the United States. It's not a disease that can be treated with antibiotics, but it can be cured with a combination of antibiotics. If you're concerned about your health, you may want 

### Out-of-Domain Evaluation

The fine-tuned model was evaluated on 10 out-of-domain questions: five unrelated to finance and five finance-related questions outside the intended financial-literacy scope.

The model failed to reliably recognize out-of-domain inputs. Instead of declining or redirecting inappropriate questions, it generally attempted to answer them. Several responses were vague, irrelevant, or factually incorrect. For example, the model produced unsupported medical information for a malaria question and speculative responses to short-term investment questions.

These results indicate that domain-specific fine-tuning improved in-domain question answering but did not provide reliable scope awareness. Therefore, a separate out-of-domain detection layer is required before deployment.

### Out-of-Domain Detection

To prevent the financial-literacy assistant from answering unrelated or unsupported questions, a lightweight semantic out-of-domain detector will be added before the FLAN-T5 model.

The detector will compare each user question with known in-domain financial-literacy questions using MiniLM sentence embeddings and cosine similarity. If the question is sufficiently similar to the financial-literacy domain, it will be passed to FLAN-T5 for answer generation. Otherwise, the system will return a short message indicating that the assistant is limited to financial-literacy questions.

A similarity threshold will be selected using the validation questions as in-domain examples and the manually created OOD questions as out-of-domain examples.

First, we will start by preparing known in-domain questions, and encode them with MiniLM.

In [19]:
TRAIN_PATH = "/kaggle/input/datasets/irakozegaddiel/fineduqa-fiqa-data/fiqa_train.csv"
VAL_PATH = "/kaggle/input/datasets/irakozegaddiel/fineduqa-fiqa-data/fiqa_validation.csv"

train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

train_questions = train_df["question_clean"].dropna().drop_duplicates().tolist()
val_questions = val_df["question_clean"].dropna().drop_duplicates().tolist()

print("Unique training questions:", len(train_questions))
print("Unique validation questions:", len(val_questions))
print("OOD questions:", len(ood_df))

Unique training questions: 1592
Unique validation questions: 199
OOD questions: 10


We will encode the 1,592 training questions, 199 validation questions, and 10 OOD questions using the same MiniLM model we used for semantic similarity.

In [20]:
exp2_model.to("cpu")
torch.cuda.empty_cache()

SEMANTIC_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
SEMANTIC_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

semantic_tokenizer = AutoTokenizer.from_pretrained(SEMANTIC_MODEL)
semantic_model = AutoModel.from_pretrained(SEMANTIC_MODEL).to(SEMANTIC_DEVICE)
semantic_model.eval()

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 384, padding_idx=0)
    (position_embeddings): Embedding(512, 384)
    (token_type_embeddings): Embedding(2, 384)
    (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-5): 6 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=384, out_features=384, bias=True)
            (key): Linear(in_features=384, out_features=384, bias=True)
            (value): Linear(in_features=384, out_features=384, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=384, out_features=384, bias=True)
            (LayerNorm): LayerNorm((384,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
    

In [21]:
# Embedding functions definition
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output.last_hidden_state
    mask = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)

def encode_questions(questions, batch_size=32):
    embeddings = []

    for i in tqdm(range(0, len(questions), batch_size), desc="Encoding questions"):
        batch = questions[i:i + batch_size]
        inputs = semantic_tokenizer(batch, padding=True, truncation=True, max_length=128, return_tensors="pt").to(SEMANTIC_DEVICE)

        with torch.no_grad():
            outputs = semantic_model(**inputs)

        batch_embeddings = mean_pooling(outputs, inputs["attention_mask"])
        batch_embeddings = F.normalize(batch_embeddings, p=2, dim=1)
        embeddings.append(batch_embeddings.cpu())

    return torch.cat(embeddings, dim=0)

In [22]:
ood_questions_text = ood_df["question"].tolist()

train_embeddings = encode_questions(train_questions)
val_embeddings = encode_questions(val_questions)
ood_embeddings = encode_questions(ood_questions_text)

print("Train embeddings:", train_embeddings.shape)
print("Validation embeddings:", val_embeddings.shape)
print("OOD embeddings:", ood_embeddings.shape)

Encoding questions:   0%|          | 0/50 [00:00<?, ?it/s]

Encoding questions:   0%|          | 0/7 [00:00<?, ?it/s]

Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

Train embeddings: torch.Size([1592, 384])
Validation embeddings: torch.Size([199, 384])
OOD embeddings: torch.Size([10, 384])


Calculate similarity to the financial domain: For each validation or OOD question, we'll compare it against all 1,592 training questions. Rather than trusting only one nearest question, we'll use the mean similarity of the top 5 closest training questions.

In [23]:
def get_domain_scores(query_embeddings, reference_embeddings, top_k=5):
    similarities = torch.mm(query_embeddings, reference_embeddings.T)
    top_scores = torch.topk(similarities, k=top_k, dim=1).values

    max_similarity = top_scores[:, 0]
    top_k_mean = top_scores.mean(dim=1)

    return max_similarity.numpy(), top_k_mean.numpy()

In [25]:
# Calculate the scores:
val_max_scores, val_domain_scores = get_domain_scores(val_embeddings, train_embeddings)
ood_max_scores, ood_domain_scores = get_domain_scores(ood_embeddings, train_embeddings)

In [26]:
# Create small analysis dataframes
val_ood_scores = pd.DataFrame({
    "question": val_questions,
    "group": "In-domain",
    "max_similarity": val_max_scores,
    "top5_mean_similarity": val_domain_scores
})

ood_scores = pd.DataFrame({
    "question": ood_questions_text,
    "group": "OOD",
    "max_similarity": ood_max_scores,
    "top5_mean_similarity": ood_domain_scores
})

domain_score_df = pd.concat([val_ood_scores, ood_scores], ignore_index=True)

print(domain_score_df.shape)

(209, 4)


Next, we will inspect the score distributions: to see whether the two groups naturally separate.

In [27]:
print("IN-DOMAIN TOP-5 SIMILARITY")
print(pd.Series(val_domain_scores).describe())

print("\nOOD TOP-5 SIMILARITY")
print(pd.Series(ood_domain_scores).describe())

IN-DOMAIN TOP-5 SIMILARITY
count    199.000000
mean       0.572933
std        0.068922
min        0.366383
25%        0.523782
50%        0.571527
75%        0.616012
max        0.812175
dtype: float64

OOD TOP-5 SIMILARITY
count    10.000000
mean      0.356507
std       0.153525
min       0.181581
25%       0.209797
50%       0.359474
75%       0.493220
max       0.567542
dtype: float64


In [28]:
# Also print every OOD question with its score
for i, row in ood_scores.iterrows():
    print("=" * 100)
    print("QUESTION:", row["question"])
    print("Max similarity:", round(row["max_similarity"], 4))
    print("Top-5 mean similarity:", round(row["top5_mean_similarity"], 4))

QUESTION: How does photosynthesis work?
Max similarity: 0.2358
Top-5 mean similarity: 0.2079
QUESTION: How do I fix an IndexError in Python?
Max similarity: 0.281
Top-5 mean similarity: 0.271
QUESTION: What causes malaria?
Max similarity: 0.1977
Top-5 mean similarity: 0.1816
QUESTION: How do I cook rice properly?
Max similarity: 0.2143
Top-5 mean similarity: 0.2024
QUESTION: Why is the sky blue?
Max similarity: 0.2541
Top-5 mean similarity: 0.2154
QUESTION: Which stock should I buy today for the highest short-term return?
Max similarity: 0.5954
Top-5 mean similarity: 0.5675
QUESTION: What options strategy should I use to maximize profit this week?
Max similarity: 0.5443
Top-5 mean similarity: 0.5095
QUESTION: How should I calculate deferred tax liabilities for a multinational corporation?
Max similarity: 0.4996
Top-5 mean similarity: 0.4562
QUESTION: Should I short Bitcoin before the next Federal Reserve announcement?
Max similarity: 0.5424
Top-5 mean similarity: 0.448
QUESTION: How do

### Remark on Semantic OOD Detection

The semantic similarity results show a clear distinction between **non-financial questions** and the financial-literacy domain. The five non-finance OOD questions produced very low top-5 similarity scores, roughly between **0.18 and 0.27**, while the in-domain validation questions had a much higher mean score of approximately **0.57**.

However, the detector is less effective for **finance-related but out-of-scope questions**. Advanced or speculative finance questions, such as short-term stock selection, options strategies, deferred tax, Bitcoin shorting, and leveraged buyout valuation, still achieved relatively high similarity scores of about **0.45–0.57** because they share financial terminology with the training data.

This indicates that semantic similarity is effective for identifying clearly unrelated domains, but it is not sufficient on its own to distinguish basic financial-literacy questions from advanced or speculative finance topics. A second scope-checking mechanism is therefore required.

In [30]:
# Inspect candidate semantic thresholds: Before choosing the first threshold, let's quantify the trade-off.

thresholds = [0.35, 0.40, 0.45, 0.50, 0.55]

non_finance_scores = ood_scores.loc[ood_scores["question"].isin(ood_questions_text[:5]), "top5_mean_similarity"].values
finance_ood_scores = ood_scores.loc[ood_scores["question"].isin(ood_questions_text[5:]), "top5_mean_similarity"].values

results = []

for threshold in thresholds:
    in_domain_accept = (val_domain_scores >= threshold).mean()
    non_finance_reject = (non_finance_scores < threshold).mean()
    finance_ood_reject = (finance_ood_scores < threshold).mean()

    results.append({
        "threshold": threshold,
        "in_domain_accept_rate": in_domain_accept,
        "non_finance_reject_rate": non_finance_reject,
        "finance_ood_reject_rate": finance_ood_reject
    })

threshold_results = pd.DataFrame(results)
display(threshold_results)

,threshold,in_domain_accept_rate,non_finance_reject_rate,finance_ood_reject_rate
0,0.35,1.000000,1.0,0.0
1,0.40,0.989950,1.0,0.0
2,0.45,0.974874,1.0,0.2
3,0.50,0.864322,1.0,0.4
4,0.55,0.608040,1.0,0.8


Next, we'll build the first OOD gate and test it

In [31]:
SEMANTIC_THRESHOLD = 0.40
TOP_K = 5

def get_question_domain_score(question):
    question_embedding = encode_questions([question])
    similarities = torch.mm(question_embedding, train_embeddings.T)
    top_scores = torch.topk(similarities, k=TOP_K, dim=1).values
    return top_scores.mean().item()

def passes_semantic_domain_check(question):
    score = get_question_domain_score(question)
    return score >= SEMANTIC_THRESHOLD, score

In [32]:
test_questions = [
    "How can I improve my credit score?",
    "How much should I keep in an emergency fund?",
    "How does photosynthesis work?",
    "What causes malaria?",
    "Which stock should I buy today?"
]

for question in test_questions:
    accepted, score = passes_semantic_domain_check(question)
    print("Question:", question)
    print("Score:", round(score, 4))
    print("Semantic check:", "PASS" if accepted else "REJECT")
    print("-" * 80)

Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

Question: How can I improve my credit score?
Score: 0.7422
Semantic check: PASS
--------------------------------------------------------------------------------


Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

Question: How much should I keep in an emergency fund?
Score: 0.6662
Semantic check: PASS
--------------------------------------------------------------------------------


Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

Question: How does photosynthesis work?
Score: 0.2079
Semantic check: REJECT
--------------------------------------------------------------------------------


Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

Question: What causes malaria?
Score: 0.1816
Semantic check: REJECT
--------------------------------------------------------------------------------


Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

Question: Which stock should I buy today?
Score: 0.5499
Semantic check: PASS
--------------------------------------------------------------------------------


Next, we'll build the second gate: Define the intended and excluded scopes

In [33]:
in_scope_topics = [
    "personal budgeting and managing everyday expenses",
    "saving money and building an emergency fund",
    "bank accounts and basic banking services",
    "credit scores and responsible use of credit",
    "personal debt and repayment of loans",
    "student loans and education expenses",
    "mobile money and everyday digital payments",
    "financial scams, fraud prevention and financial safety",
    "beginner investing and basic investment concepts",
    "basic insurance for individuals and families",
    "personal financial planning and financial goals"
]

out_of_scope_topics = [
    "choosing specific stocks for short-term profit",
    "day trading and short-term market speculation",
    "advanced options and derivatives trading strategies",
    "short selling stocks or cryptocurrencies",
    "cryptocurrency speculation and trading strategies",
    "corporate accounting and corporate taxation",
    "multinational corporate tax calculations",
    "leveraged buyout models and advanced company valuation",
    "investment banking and complex corporate finance"
]

In [34]:
# Encode the scope descriptions
in_scope_embeddings = encode_questions(in_scope_topics)
out_of_scope_embeddings = encode_questions(out_of_scope_topics)

print("In-scope embeddings:", in_scope_embeddings.shape)
print("Out-of-scope embeddings:", out_of_scope_embeddings.shape)

Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

In-scope embeddings: torch.Size([11, 384])
Out-of-scope embeddings: torch.Size([9, 384])


In [35]:
# Create the scope scoring function
def get_scope_scores(question):
    question_embedding = encode_questions([question])
    in_scope_score = torch.mm(question_embedding, in_scope_embeddings.T).max().item()
    out_scope_score = torch.mm(question_embedding, out_of_scope_embeddings.T).max().item()
    return in_scope_score, out_scope_score

In [36]:
# Test it on representative questions
scope_test_questions = [
    "How can I improve my credit score?",
    "How much should I keep in an emergency fund?",
    "What is an index fund?",
    "Which stock should I buy today for the highest short-term return?",
    "What options strategy should I use to maximize profit this week?",
    "How should I calculate deferred tax liabilities for a multinational corporation?",
    "Should I short Bitcoin before the next Federal Reserve announcement?",
    "How do I value a company using a leveraged buyout model?"
]

for question in scope_test_questions:
    in_score, out_score = get_scope_scores(question)

    print("=" * 100)
    print("Question:", question)
    print("In-scope similarity:", round(in_score, 4))
    print("Out-of-scope similarity:", round(out_score, 4))
    print("Difference:", round(in_score - out_score, 4))

Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

Question: How can I improve my credit score?
In-scope similarity: 0.6008
Out-of-scope similarity: 0.1789
Difference: 0.4219


Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

Question: How much should I keep in an emergency fund?
In-scope similarity: 0.7121
Out-of-scope similarity: 0.2312
Difference: 0.4809


Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

...

Question: How should I calculate deferred tax liabilities for a multinational corporation?
In-scope similarity: 0.2432
Out-of-scope similarity: 0.7698
Difference: -0.5267


Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

Question: Should I short Bitcoin before the next Federal Reserve announcement?
In-scope similarity: 0.3243
Out-of-scope similarity: 0.5325
Difference: -0.2082


Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

Question: How do I value a company using a leveraged buyout model?
In-scope similarity: 0.2447
Out-of-scope similarity: 0.8632
Difference: -0.6186


These results are very promising. The intended financial-literacy questions all have positive margins, while every advanced/speculative finance example has a negative margin. However, we should not choose the final rule from only these eight examples. Next, we will test the scope check across all 199 in-domain validation questions and the 5 finance OOD questions we already created.

In [37]:
def get_scope_margins(question_embeddings):
    in_scores = torch.mm(question_embeddings, in_scope_embeddings.T).max(dim=1).values
    out_scores = torch.mm(question_embeddings, out_of_scope_embeddings.T).max(dim=1).values
    margins = in_scores - out_scores
    return in_scores.numpy(), out_scores.numpy(), margins.numpy()

In [38]:
val_in_scores, val_out_scores, val_scope_margins = get_scope_margins(val_embeddings)

finance_ood_embeddings = ood_embeddings[5:]
finance_ood_in_scores, finance_ood_out_scores, finance_ood_margins = get_scope_margins(finance_ood_embeddings)

In [39]:
print("IN-DOMAIN VALIDATION MARGINS")
print(pd.Series(val_scope_margins).describe())

print("\nFINANCE OOD MARGINS")
print(pd.Series(finance_ood_margins).describe())

IN-DOMAIN VALIDATION MARGINS
count    199.000000
mean       0.099910
std        0.155363
min       -0.298279
25%       -0.012213
50%        0.095972
75%        0.199989
max        0.552496
dtype: float64

FINANCE OOD MARGINS
count    5.000000
mean    -0.402666
std      0.182556
min     -0.618551
25%     -0.526650
50%     -0.437064
75%     -0.222843
max     -0.208221
dtype: float64


This also tells us that using 0 as the threshold would be a bad idea: roughly a quarter of legitimate validation questions already have margins below about -0.012, so a 0 threshold would reject too many valid questions. For that, let's test better candidate thresholds.

In [40]:
scope_thresholds = [-0.25, -0.20, -0.15, -0.10, -0.05, 0.00]
scope_results = []

for threshold in scope_thresholds:
    in_domain_accept_rate = (val_scope_margins >= threshold).mean()
    finance_ood_reject_rate = (finance_ood_margins < threshold).mean()

    scope_results.append({
        "threshold": threshold,
        "in_domain_accept_rate": in_domain_accept_rate,
        "finance_ood_reject_rate": finance_ood_reject_rate
    })

scope_threshold_results = pd.DataFrame(scope_results)
display(scope_threshold_results)

,threshold,in_domain_accept_rate,finance_ood_reject_rate
0,-0.25,0.989950,0.6
1,-0.20,0.984925,1.0
2,-0.15,0.944724,1.0
3,-0.10,0.914573,1.0
4,-0.05,0.829146,1.0
5,0.00,0.728643,1.0


At -0.20, the second gate accepts 98.49% of legitimate validation questions, and rejects 100% of the advanced/speculative finance OOD questions.

So, for the next step we will finally combine both checks into one detector.

In [41]:
SEMANTIC_THRESHOLD = 0.40
SCOPE_THRESHOLD = -0.20
TOP_K = 5

def classify_question_scope(question):
    question_embedding = encode_questions([question])

    similarities = torch.mm(question_embedding, train_embeddings.T)
    top_scores = torch.topk(similarities, k=TOP_K, dim=1).values
    domain_score = top_scores.mean().item()

    if domain_score < SEMANTIC_THRESHOLD:
        return {
            "in_domain": False,
            "reason": "non_financial",
            "domain_score": domain_score,
            "scope_margin": None
        }

    in_scope_score = torch.mm(question_embedding, in_scope_embeddings.T).max().item()
    out_scope_score = torch.mm(question_embedding, out_of_scope_embeddings.T).max().item()
    scope_margin = in_scope_score - out_scope_score

    if scope_margin < SCOPE_THRESHOLD:
        return {
            "in_domain": False,
            "reason": "out_of_scope_finance",
            "domain_score": domain_score,
            "scope_margin": scope_margin
        }

    return {
        "in_domain": True,
        "reason": "in_scope",
        "domain_score": domain_score,
        "scope_margin": scope_margin
    }

In [42]:
final_detector_test = [
    "How can I improve my credit score?",
    "How much should I keep in an emergency fund?",
    "What is an index fund?",
    "How does photosynthesis work?",
    "What causes malaria?",
    "Which stock should I buy today for the highest short-term return?",
    "What options strategy should I use to maximize profit this week?",
    "How should I calculate deferred tax liabilities for a multinational corporation?",
    "Should I short Bitcoin before the next Federal Reserve announcement?",
    "How do I value a company using a leveraged buyout model?"
]

for question in final_detector_test:
    result = classify_question_scope(question)

    print("=" * 100)
    print("Question:", question)
    print("Decision:", "ACCEPT" if result["in_domain"] else "REJECT")
    print("Reason:", result["reason"])
    print("Domain score:", round(result["domain_score"], 4))

    if result["scope_margin"] is not None:
        print("Scope margin:", round(result["scope_margin"], 4))

Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

Question: How can I improve my credit score?
Decision: ACCEPT
Reason: in_scope
Domain score: 0.7422
Scope margin: 0.4219


Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

Question: How much should I keep in an emergency fund?
Decision: ACCEPT
Reason: in_scope
Domain score: 0.6662
Scope margin: 0.4809


Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

...

Question: How should I calculate deferred tax liabilities for a multinational corporation?
Decision: REJECT
Reason: out_of_scope_finance
Domain score: 0.4562
Scope margin: -0.5267


Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

Question: Should I short Bitcoin before the next Federal Reserve announcement?
Decision: REJECT
Reason: out_of_scope_finance
Domain score: 0.448
Scope margin: -0.2082


Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

Question: How do I value a company using a leveraged buyout model?
Decision: REJECT
Reason: out_of_scope_finance
Domain score: 0.5055
Scope margin: -0.6186


### Remark on Final OOD Evaluation

The initial OOD questions were used to design and tune the detector thresholds, so they should not be reused as the final evaluation set. To obtain a fairer estimate of performance, the final OOD evaluation will use the unseen test questions as in-domain examples and a new set of OOD questions that were not used during threshold selection.

In [43]:
# First load the test set and keep only unique questions
TEST_PATH = "/kaggle/input/datasets/irakozegaddiel/fineduqa-fiqa-data/fiqa_test.csv"

test_df = pd.read_csv(TEST_PATH)
test_questions = test_df["question_clean"].dropna().drop_duplicates().tolist()

print("Unique test questions:", len(test_questions))

Unique test questions: 200


In [44]:
# Create a new OOD evaluation set that we have not used during threshold selection
heldout_ood = [
    {"category": "Non-finance", "question": "What is the difference between mitosis and meiosis?"},
    {"category": "Non-finance", "question": "How can I sort a Python list alphabetically?"},
    {"category": "Non-finance", "question": "Why do earthquakes occur?"},
    {"category": "Non-finance", "question": "How do vaccines help the immune system?"},
    {"category": "Non-finance", "question": "What temperature should I use to bake a cake?"},
    {"category": "Non-finance", "question": "How does a combustion engine work?"},
    {"category": "Non-finance", "question": "What are the main causes of climate change?"},
    {"category": "Non-finance", "question": "How do I create a database table in SQL?"},
    {"category": "Non-finance", "question": "Why do humans need sleep?"},
    {"category": "Non-finance", "question": "How does gravity affect objects on Earth?"},
    {"category": "Out-of-scope finance", "question": "Which technology stock is most likely to outperform the market next month?"},
    {"category": "Out-of-scope finance", "question": "How should I construct a delta-neutral options portfolio?"},
    {"category": "Out-of-scope finance", "question": "How do I calculate the weighted average cost of capital for an acquisition?"},
    {"category": "Out-of-scope finance", "question": "Should I use leverage to trade cryptocurrency futures?"},
    {"category": "Out-of-scope finance", "question": "How do I build a discounted cash flow valuation model for a public company?"},
    {"category": "Out-of-scope finance", "question": "Which forex pair should I trade this week?"},
    {"category": "Out-of-scope finance", "question": "How should a corporation structure an international tax avoidance strategy?"},
    {"category": "Out-of-scope finance", "question": "What gamma scalping strategy should an options trader use?"},
    {"category": "Out-of-scope finance", "question": "How do I model synergies in a merger and acquisition transaction?"},
    {"category": "Out-of-scope finance", "question": "Should I short a company's shares before its earnings announcement?"}
]

heldout_ood_df = pd.DataFrame(heldout_ood)

print("Held-out OOD questions:", len(heldout_ood_df))

Held-out OOD questions: 20


In [46]:
# Encode the held-out sets
test_embeddings = encode_questions(test_questions)
heldout_ood_embeddings = encode_questions(heldout_ood_df["question"].tolist())

print("Test embeddings:", test_embeddings.shape)
print("Held-out OOD embeddings:", heldout_ood_embeddings.shape)

Encoding questions:   0%|          | 0/7 [00:00<?, ?it/s]

Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

Test embeddings: torch.Size([200, 384])
Held-out OOD embeddings: torch.Size([20, 384])


In [47]:
# Evaluate the complete two-stage detector

def classify_embeddings(question_embeddings):
    similarities = torch.mm(question_embeddings, train_embeddings.T)
    domain_scores = torch.topk(similarities, k=TOP_K, dim=1).values.mean(dim=1)

    in_scores = torch.mm(question_embeddings, in_scope_embeddings.T).max(dim=1).values
    out_scores = torch.mm(question_embeddings, out_of_scope_embeddings.T).max(dim=1).values
    scope_margins = in_scores - out_scores

    accepted = (domain_scores >= SEMANTIC_THRESHOLD) & (scope_margins >= SCOPE_THRESHOLD)
    return accepted.numpy(), domain_scores.numpy(), scope_margins.numpy()

test_accepted, test_domain_scores, test_scope_margins = classify_embeddings(test_embeddings)
ood_accepted, ood_domain_scores, ood_scope_margins = classify_embeddings(heldout_ood_embeddings)

in_domain_accept_rate = test_accepted.mean()
overall_ood_reject_rate = (~ood_accepted).mean()

non_finance_mask = heldout_ood_df["category"] == "Non-finance"
finance_ood_mask = heldout_ood_df["category"] == "Out-of-scope finance"

non_finance_reject_rate = (~ood_accepted[non_finance_mask]).mean()
finance_ood_reject_rate = (~ood_accepted[finance_ood_mask]).mean()

print("In-domain test acceptance:", round(in_domain_accept_rate, 4))
print("Overall OOD rejection:", round(overall_ood_reject_rate, 4))
print("Non-finance rejection:", round(non_finance_reject_rate, 4))
print("Advanced-finance OOD rejection:", round(finance_ood_reject_rate, 4))

In-domain test acceptance: 0.98
Overall OOD rejection: 0.95
Non-finance rejection: 1.0
Advanced-finance OOD rejection: 0.9


We notice that the detector generalizes well to unseen data, not just the examples used to choose the thresholds. Therefore, the remaining errors are small enough that we should inspect them rather than change the thresholds immediately.

In [48]:
# Let's investigate the 4 legitimate test questions that were wrongly rejected:
test_results_df = pd.DataFrame({
    "question": test_questions,
    "accepted": test_accepted,
    "domain_score": test_domain_scores,
    "scope_margin": test_scope_margins
})

false_rejections = test_results_df[~test_results_df["accepted"]]

print("Incorrectly rejected in-domain questions:", len(false_rejections))

for i, row in false_rejections.iterrows():
    print("=" * 100)
    print("QUESTION:", row["question"])
    print("Domain score:", round(row["domain_score"], 4))
    print("Scope margin:", round(row["scope_margin"], 4))

Incorrectly rejected in-domain questions: 4
QUESTION: How are various types of income taxed differently in the USA?
Domain score: 0.5284
Scope margin: -0.2431
QUESTION: Tax implications of diversification
Domain score: 0.5876
Scope margin: -0.2203
QUESTION: Do I have to pay taxes on income from my website or profits?
Domain score: 0.5916
Scope margin: -0.2476
QUESTION: Apartment Security Deposit refunds in Maryland
Domain score: 0.3981
Scope margin: 0.1318


In [49]:
# Then inspect the OOD question that incorrectly passed
heldout_ood_results = heldout_ood_df.copy()
heldout_ood_results["accepted"] = ood_accepted
heldout_ood_results["domain_score"] = ood_domain_scores
heldout_ood_results["scope_margin"] = ood_scope_margins

false_acceptances = heldout_ood_results[heldout_ood_results["accepted"]]

print("Incorrectly accepted OOD questions:", len(false_acceptances))

for i, row in false_acceptances.iterrows():
    print("=" * 100)
    print("CATEGORY:", row["category"])
    print("QUESTION:", row["question"])
    print("Domain score:", round(row["domain_score"], 4))
    print("Scope margin:", round(row["scope_margin"], 4))

Incorrectly accepted OOD questions: 1
CATEGORY: Out-of-scope finance
QUESTION: How do I build a discounted cash flow valuation model for a public company?
Domain score: 0.4689
Scope margin: -0.1492


### OOD Detector Error Analysis

The held-out evaluation achieved a 98% acceptance rate on the FiQA test questions and a 95% rejection rate on unseen OOD questions. All non-financial OOD questions were rejected, while 9 of 10 advanced-finance questions were rejected.

Four FiQA test questions were rejected. Three concerned jurisdiction-specific US taxation and one concerned Maryland security-deposit law. These topics are outside the narrower financial-literacy scope defined for the deployed assistant, despite remaining in the filtered FiQA dataset. The only clear OOD false acceptance was a discounted cash flow valuation question, showing that advanced finance questions close to the decision boundary can occasionally pass the detector.

### Connect the detector to FLAN-T5

In [50]:
# Put MiniLM on CPU and FLAN-T5 on GPU
OOD_DEVICE = torch.device("cpu")
GENERATION_DEVICE = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

semantic_model.to(OOD_DEVICE)
exp2_model.to(GENERATION_DEVICE)

semantic_model.eval()
exp2_model.eval()

SEMANTIC_DEVICE = OOD_DEVICE

print("OOD detector device:", OOD_DEVICE)
print("FLAN-T5 device:", GENERATION_DEVICE)

OOD detector device: cpu
FLAN-T5 device: cuda:0


In [51]:
# Create the final assistant function
def financial_assistant(question):
    result = classify_question_scope(question)

    if not result["in_domain"]:
        if result["reason"] == "non_financial":
            answer = "This assistant is designed to answer financial-literacy questions. Please ask a question related to personal finance, saving, budgeting, banking, credit, debt, insurance, or beginner investing."
        else:
            answer = "This question is outside the scope of this financial-literacy assistant. Please ask about personal finance topics such as budgeting, saving, banking, credit, debt, insurance, or beginner investing."

        return {
            "answer": answer,
            "status": "rejected",
            "reason": result["reason"],
            "domain_score": result["domain_score"],
            "scope_margin": result["scope_margin"]
        }

    answer = generate_answer_with_controls(exp2_model, question, GENERATION_DEVICE)

    return {
        "answer": answer,
        "status": "accepted",
        "reason": result["reason"],
        "domain_score": result["domain_score"],
        "scope_margin": result["scope_margin"]
    }

Finally, let's test the complete pipeline

In [52]:
pipeline_test_questions = [
    "How can I improve my credit score?",
    "How much should I keep in an emergency fund?",
    "What is an index fund?",
    "How does photosynthesis work?",
    "What causes malaria?",
    "Which stock should I buy today for the highest short-term return?",
    "How do I value a company using a leveraged buyout model?"
]

for question in pipeline_test_questions:
    result = financial_assistant(question)

    print("=" * 100)
    print("QUESTION:")
    print(question)

    print("\nSTATUS:")
    print(result["status"])

    print("\nREASON:")
    print(result["reason"])

    print("\nANSWER:")
    print(result["answer"])

    print("\nDomain score:", round(result["domain_score"], 4))

    if result["scope_margin"] is not None:
        print("Scope margin:", round(result["scope_margin"], 4))

Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

QUESTION:
How can I improve my credit score?

STATUS:
accepted

REASON:
in_scope

ANSWER:
If you're looking for a way to improve your credit score, there are a number of ways you can do that. The first is to make sure that you have a good credit score. If you don't, then you'll need to get a credit report from the credit bureaus. The second is to find out how much you can afford to pay off your debts. You'll also need to know how much money you can spend on things like gas, food, etc. The third is to look at your credit report and see if you can improve your score.

Domain score: 0.7422
Scope margin: 0.4219


Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

QUESTION:
How much should I keep in an emergency fund?

STATUS:
accepted

REASON:
in_scope

ANSWER:
If you have a large amount of money in your emergency fund, it's important to keep it in a safe place. If you don't have enough money, you may want to consider investing in an emergency fund.

Domain score: 0.6662
Scope margin: 0.4809


Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

...

QUESTION:
What causes malaria?

STATUS:
rejected

REASON:
non_financial

ANSWER:
This assistant is designed to answer financial-literacy questions. Please ask a question related to personal finance, saving, budgeting, banking, credit, debt, insurance, or beginner investing.

Domain score: 0.1816


Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

QUESTION:
Which stock should I buy today for the highest short-term return?

STATUS:
rejected

REASON:
out_of_scope_finance

ANSWER:
This question is outside the scope of this financial-literacy assistant. Please ask about personal finance topics such as budgeting, saving, banking, credit, debt, insurance, or beginner investing.

Domain score: 0.5675
Scope margin: -0.4371


Encoding questions:   0%|          | 0/1 [00:00<?, ?it/s]

QUESTION:
How do I value a company using a leveraged buyout model?

STATUS:
rejected

REASON:
out_of_scope_finance

ANSWER:
This question is outside the scope of this financial-literacy assistant. Please ask about personal finance topics such as budgeting, saving, banking, credit, debt, insurance, or beginner investing.

Domain score: 0.5055
Scope margin: -0.6186


In [53]:
# Save the OOD detector

OOD_ARTIFACT_PATH = "/kaggle/working/ood_detector_artifacts.pt"
OOD_CONFIG_PATH = "/kaggle/working/ood_detector_config.json"

ood_artifacts = {
    "train_embeddings": train_embeddings,
    "in_scope_embeddings": in_scope_embeddings,
    "out_of_scope_embeddings": out_of_scope_embeddings,
    "train_questions": train_questions,
    "in_scope_topics": in_scope_topics,
    "out_of_scope_topics": out_of_scope_topics
}

torch.save(ood_artifacts, OOD_ARTIFACT_PATH)

ood_config = {
    "semantic_model": SEMANTIC_MODEL,
    "semantic_threshold": SEMANTIC_THRESHOLD,
    "scope_threshold": SCOPE_THRESHOLD,
    "top_k": TOP_K
}

with open(OOD_CONFIG_PATH, "w") as f:
    json.dump(ood_config, f, indent=4)

print("OOD artifacts saved to:", OOD_ARTIFACT_PATH)
print("OOD configuration saved to:", OOD_CONFIG_PATH)

OOD artifacts saved to: /kaggle/working/ood_detector_artifacts.pt
OOD configuration saved to: /kaggle/working/ood_detector_config.json


In [55]:
# Save the final OOD evaluation results
ood_metrics = pd.DataFrame({
    "Metric": [
        "In-domain test acceptance",
        "Overall OOD rejection",
        "Non-finance rejection",
        "Advanced-finance OOD rejection"
    ],
    "Rate": [
        in_domain_accept_rate,
        overall_ood_reject_rate,
        non_finance_reject_rate,
        finance_ood_reject_rate
    ]
})

ood_metrics.to_csv("/kaggle/working/final_ood_metrics.csv", index=False)
test_results_df.to_csv("/kaggle/working/ood_in_domain_test_results.csv", index=False)
heldout_ood_results.to_csv("/kaggle/working/ood_heldout_results.csv", index=False)

display(ood_metrics)

print("Saved final OOD evaluation files.")

,Metric,Rate
0,In-domain test acceptance,0.98
1,Overall OOD rejection,0.95
2,Non-finance rejection,1.00
3,Advanced-finance OOD rejection,0.90


Saved final OOD evaluation files.


In [56]:
# Verify the saved detector can be reloaded

loaded_artifacts = torch.load("/kaggle/working/ood_detector_artifacts.pt", weights_only=False)

with open("/kaggle/working/ood_detector_config.json", "r") as f:
    loaded_config = json.load(f)

print("Training embeddings:", loaded_artifacts["train_embeddings"].shape)
print("In-scope embeddings:", loaded_artifacts["in_scope_embeddings"].shape)
print("Out-of-scope embeddings:", loaded_artifacts["out_of_scope_embeddings"].shape)

print("\nSemantic threshold:", loaded_config["semantic_threshold"])
print("Scope threshold:", loaded_config["scope_threshold"])
print("Top K:", loaded_config["top_k"])
print("Semantic model:", loaded_config["semantic_model"])

Training embeddings: torch.Size([1592, 384])
In-scope embeddings: torch.Size([11, 384])
Out-of-scope embeddings: torch.Size([9, 384])

Semantic threshold: 0.4
Scope threshold: -0.2
Top K: 5
Semantic model: sentence-transformers/all-MiniLM-L6-v2
